# 01. Importing Libraries

In [1]:
import os
import math
import random
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import scipy
import numpy as np
import pandas as pd

# 02. Importing Data

In [2]:
data = pd.read_csv("../Data/Clean_data/merged_web_data_group_test.csv")

In [3]:
data.shape

(317235, 7)

In [4]:
data.head()

,client_id,visitor_id,visit_id,process_step,date_time,source,group_test
0,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:22:17,pt1,test
1,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:23:09,pt1,test
2,7338123,612065484_94198474375,100019538_17884295066_43909,start,2017-04-09 16:20:56,pt1,test
3,7338123,612065484_94198474375,100019538_17884295066_43909,step_1,2017-04-09 16:21:12,pt1,test
4,7338123,612065484_94198474375,100019538_17884295066_43909,step_2,2017-04-09 16:21:21,pt1,test


In [5]:
data['process_step'].value_counts()

process_step
start      101153
step_1      68210
step_2      56672
step_3      48264
confirm     42936
Name: count, dtype: int64

In [ ]:
data.dtypes

client_id       int64
visitor_id        str
visit_id          str
process_step      str
date_time         str
source            str
group_test        str
dtype: object

In [11]:
# converting date_time to datetime format
data['date_time'] = pd.to_datetime(data['date_time'], format='%Y-%m-%d %H:%M:%S')

In [12]:
data.dtypes

client_id                int64
visitor_id                 str
visit_id                   str
process_step               str
date_time       datetime64[us]
source                     str
group_test                 str
dtype: object

In [13]:
data.head()

,client_id,visitor_id,visit_id,process_step,date_time,source,group_test
0,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:22:17,pt1,test
1,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:23:09,pt1,test
2,7338123,612065484_94198474375,100019538_17884295066_43909,start,2017-04-09 16:20:56,pt1,test
3,7338123,612065484_94198474375,100019538_17884295066_43909,step_1,2017-04-09 16:21:12,pt1,test
4,7338123,612065484_94198474375,100019538_17884295066_43909,step_2,2017-04-09 16:21:21,pt1,test


In [14]:
# Mapping process steps to numeric values for easier analysis
step_order = {'start': 0, 'step_1': 1, 'step_2': 2, 'step_3': 3, 'confirm': 4}

data['step_num'] = data['process_step'].map(step_order)

In [15]:
# checking the new column
data

,client_id,visitor_id,visit_id,process_step,date_time,source,group_test,step_num
0,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:22:17,pt1,test,4
1,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:23:09,pt1,test,4
2,7338123,612065484_94198474375,100019538_17884295066_43909,start,2017-04-09 16:20:56,pt1,test,0
3,7338123,612065484_94198474375,100019538_17884295066_43909,step_1,2017-04-09 16:21:12,pt1,test,1
4,7338123,612065484_94198474375,100019538_17884295066_43909,step_2,2017-04-09 16:21:21,pt1,test,2
...,...,...,...,...,...,...,...,...
317230,6627522,730634087_44272418812,999988789_76411676596_272843,start,2017-04-21 23:49:11,pt1,test,0
317231,6627522,730634087_44272418812,999988789_76411676596_272843,step_1,2017-04-21 23:49:22,pt1,test,1
317232,6627522,730634087_44272418812,999988789_76411676596_272843,step_2,2017-04-21 23:50:16,pt1,test,2
317233,6627522,730634087_44272418812,999988789_76411676596_272843,step_1,2017-04-21 23:51:00,pt1,test,1


### KPI 2: time spent on each step

In [ ]:
# creating new column of the time of the next step for each visit
data['time_next'] = data.groupby('visit_id')['date_time'].shift(-1)

# calculating time spent on each step in seconds
data['time_spent'] = (data['time_next'] - data['date_time']).dt.total_seconds()

# remonving rows with time spent greater than 30 minutes (1800 seconds) as they are likely outliers
data_time = data[data['time_spent'] < 1800]

In [ ]:
# checking creation of new columns
data

,client_id,visitor_id,visit_id,process_step,date_time,source,group_test,step_num,time_next,time_spent
0,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:22:17,pt1,test,4,2017-04-26 13:23:09,52.0
1,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:23:09,pt1,test,4,NaT,NaN
2,7338123,612065484_94198474375,100019538_17884295066_43909,start,2017-04-09 16:20:56,pt1,test,0,2017-04-09 16:21:12,16.0
3,7338123,612065484_94198474375,100019538_17884295066_43909,step_1,2017-04-09 16:21:12,pt1,test,1,2017-04-09 16:21:21,9.0
4,7338123,612065484_94198474375,100019538_17884295066_43909,step_2,2017-04-09 16:21:21,pt1,test,2,2017-04-09 16:21:35,14.0
...,...,...,...,...,...,...,...,...,...,...
317230,6627522,730634087_44272418812,999988789_76411676596_272843,start,2017-04-21 23:49:11,pt1,test,0,2017-04-21 23:49:22,11.0
317231,6627522,730634087_44272418812,999988789_76411676596_272843,step_1,2017-04-21 23:49:22,pt1,test,1,2017-04-21 23:50:16,54.0
317232,6627522,730634087_44272418812,999988789_76411676596_272843,step_2,2017-04-21 23:50:16,pt1,test,2,2017-04-21 23:51:00,44.0
317233,6627522,730634087_44272418812,999988789_76411676596_272843,step_1,2017-04-21 23:51:00,pt1,test,1,2017-04-21 23:51:09,9.0


In [ ]:
# creation of a new dataset focused on group, steps and avg time spent on each step
time_kpi = data_time.groupby(['group_test', 'process_step'])['time_spent'].mean().round(2).reset_index()
time_kpi.columns = ['group', 'process_step', 'avg_time_seconds']

# creating new column for average time in minutes for easier interpretation
time_kpi['avg_time_minutes'] = (time_kpi['avg_time_seconds'] / 60).round(2)

# creating new column for step number to sort the steps in the correct order per group
time_kpi['step_num'] = time_kpi['process_step'].map(step_order)
time_kpi = time_kpi.sort_values(['group', 'step_num'])

In [ ]:
# checking new dataframe for KPI 2
time_kpi

,group,process_step,avg_time_seconds,avg_time_minutes,step_num
1,control,start,60.91,1.02,0
2,control,step_1,47.46,0.79,1
3,control,step_2,90.21,1.50,2
4,control,step_3,133.30,2.22,3
0,control,confirm,156.16,2.60,4
6,test,start,56.24,0.94,0
7,test,step_1,58.48,0.97,1
8,test,step_2,88.13,1.47,2
9,test,step_3,124.83,2.08,3
5,test,confirm,221.59,3.69,4


### KPI 3: error rate (users going back to a previous step)

In [21]:
# creating new column to identify if there are any errors in the process (going back to a previous step)
data['prev_step_num'] = data.groupby('visit_id')['step_num'].shift(1)

# creating label for error if the current step number is less than the previous step number (indicating a step back)
data['is_error'] = data['step_num'] < data['prev_step_num']


In [ ]:
# checking creation of new columns
data.head()

,client_id,visitor_id,visit_id,process_step,date_time,source,group_test,step_num,time_next,time_spent,prev_step_num,is_error
0,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:22:17,pt1,test,4,2017-04-26 13:23:09,52.0,NaN,False
1,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:23:09,pt1,test,4,NaT,NaN,4.0,False
2,7338123,612065484_94198474375,100019538_17884295066_43909,start,2017-04-09 16:20:56,pt1,test,0,2017-04-09 16:21:12,16.0,NaN,False
3,7338123,612065484_94198474375,100019538_17884295066_43909,step_1,2017-04-09 16:21:12,pt1,test,1,2017-04-09 16:21:21,9.0,0.0,False
4,7338123,612065484_94198474375,100019538_17884295066_43909,step_2,2017-04-09 16:21:21,pt1,test,2,2017-04-09 16:21:35,14.0,1.0,False


In [22]:
# creating new dataset focused on group, steps and error rate
error_kpi = data.groupby(['group_test', 'process_step']).agg(
    total_visits=('visit_id', 'count'),
    errors=('is_error', 'sum')
).reset_index()

# calculating error rate as percentage of errors out of total visits for each group and step
error_kpi['error_rate'] = (error_kpi['errors'] / error_kpi['total_visits'] * 100).round(2)

# creating new column for step number to sort the steps in the correct order per group
error_kpi['step_num'] = error_kpi['process_step'].map(step_order)
error_kpi = error_kpi.sort_values(['group_test', 'step_num'])

In [25]:
# checking new dataframe for KPI 3
# dataframe shows that, 10% of the times start was visited, users came there back from another step
error_kpi

,group_test,process_step,total_visits,errors,error_rate,step_num
1,control,start,45380,4932,10.87,0
2,control,step_1,29544,2303,7.80,1
3,control,step_2,25773,2364,9.17,2
4,control,step_3,22503,101,0.45,3
0,control,confirm,17336,0,0.00,4
6,test,start,55773,10621,19.04,0
7,test,step_1,38666,3414,8.83,1
8,test,step_2,30899,2283,7.39,2
9,test,step_3,25761,22,0.09,3
5,test,confirm,25600,0,0.00,4


In [26]:
# checking the error based on the step that people left from
step_order_reversed = {0: 'start', 1: 'step_1', 2: 'step_2', 3: 'step_3', 4: 'confirm'}
data['prev_process_step'] = data['prev_step_num'].map(step_order_reversed)

In [27]:
data.head()

,client_id,visitor_id,visit_id,process_step,date_time,source,group_test,step_num,time_next,time_spent,prev_step_num,is_error,prev_process_step
0,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:22:17,pt1,test,4,2017-04-26 13:23:09,52.0,NaN,False,NaN
1,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:23:09,pt1,test,4,NaT,NaN,4.0,False,confirm
2,7338123,612065484_94198474375,100019538_17884295066_43909,start,2017-04-09 16:20:56,pt1,test,0,2017-04-09 16:21:12,16.0,NaN,False,NaN
3,7338123,612065484_94198474375,100019538_17884295066_43909,step_1,2017-04-09 16:21:12,pt1,test,1,2017-04-09 16:21:21,9.0,0.0,False,start
4,7338123,612065484_94198474375,100019538_17884295066_43909,step_2,2017-04-09 16:21:21,pt1,test,2,2017-04-09 16:21:35,14.0,1.0,False,step_1


In [28]:
# creating new dataset grouping by the step users LEFT from (not where they landed)
error_kpi_from = data.groupby(['group_test', 'prev_process_step']).agg(
    total_visits=('visit_id', 'count'),
    errors=('is_error', 'sum')
).reset_index()

# calculating error rate as percentage
error_kpi_from['error_rate_from'] = (error_kpi_from['errors'] / error_kpi_from['total_visits'] * 100).round(2)

# creating new column for step number to sort the steps in the correct order per group
error_kpi_from['step_num'] = error_kpi_from['prev_process_step'].map(step_order)
error_kpi_from = error_kpi_from.sort_values(['group_test', 'step_num'])

In [30]:
# checking new dataframe for KPI 3
# dataframe shows that 9.57 of control on confirm went back to steps before
error_kpi_from

,group_test,prev_process_step,total_visits,errors,error_rate_from,step_num
1,control,start,35741,0,0.00,0
2,control,step_1,26046,2493,9.57,1
3,control,step_2,24316,2167,8.91,2
4,control,step_3,20281,4247,20.94,3
0,control,confirm,2032,793,39.03,4
6,test,start,46326,0,0.00,0
7,test,step_1,35535,6410,18.04,1
8,test,step_2,29576,4783,16.17,2
9,test,step_3,23984,4744,19.78,3
5,test,confirm,4193,403,9.61,4
